In [1]:
## Step 2: Imports

In [2]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_ollama import ChatOllama

base_url = "http://localhost:11434"
model = "llama3.2"

llm = ChatOllama(base_url = base_url, model = model)
llm

ChatOllama(model='llama3.2', base_url='http://localhost:11434')

## Step 3 : Create Memory (Checkpointer) 


In [3]:
checkpointer = InMemorySaver()

## Step 5 : Define Chatbot Node

In [4]:
def chatbot(state: MessagesState):
    return {
        "messages" : [llm.invoke(state["messages"])]
    }

## step 6 : Build Graph

In [5]:
builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)
builder.set_entry_point("chatbot")

graph = builder.compile(checkpointer = checkpointer)

## step 7 : Defie Thread (Memory Key)

In [6]:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

## step 8 : First Message

In [7]:
result1 = graph.invoke(
    {"messages": [("user", "Hi")]},
    config
)
result1["messages"][-1].content

'How can I help you today?'

## Step 9 : Second Message (Memory works) 

In [9]:
result2 = graph.invoke(
    {"messages": [("user", "My name is aditya")]},
    config
)
result2["messages"][-1].content

"It seems we've already established that! Hi again, Aditya! What's been new with you lately?"

## Step 10 : Third message

In [10]:
result3 = graph.invoke(
    {"messages": [("user", "What is my name")]},
    config
)
result3["messages"][-1].content

"Your name is Aditya. We've mentioned it a few times already! Would you like to talk about something specific or just have a casual chat?"